# Kaggle × Colab validation

Template for checking experiments on Colab without using a Kaggle submission. It runs `competitions/<COMPETITION>/validate.py`, which each competition provides.

Set `KAGGLE_API_TOKEN` in 🔑 Secrets, edit the options, then run all. Adapters are read from `<DRIVE_DIR>/<COMPETITION>/<exp>/` and results go to `<DRIVE_DIR>/<COMPETITION>/validate/`. Start with `--num-tasks 2` to check the setup and speed, then remove it. Running again continues from the results on Drive. With `DISCONNECT_WHEN_DONE`, the runtime is released when the run ends, so leaving it unattended does not use compute units.


In [ ]:
# @title Options
COMPETITION = "gemma-4-developer-agent"  # @param {type:"string"}
# comma-separated; the first is the baseline
EXPS = "exp001,exp005"  # @param {type:"string"}
ARGS = ""  # @param {type:"string"}
# folder in My Drive with one folder per competition: <DRIVE_DIR>/<COMPETITION>/<EXP>/
DRIVE_DIR = "Colab Notebooks/Kaggle"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
# release the GPU when the run ends (success or failure) so it stops using compute units
DISCONNECT_WHEN_DONE = True  # @param {type:"boolean"}


In [ ]:
# @title Setup
import importlib
import os
import sys

from google.colab import drive, userdata

REPO = "Moriyuki-S/kaggle_ws"
REPO_DIR = "/content/kaggle_ws"

for k in ["KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY", "GITHUB_TOKEN"]:
    try:
        os.environ[k] = userdata.get(k)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        pass

# results and adapters live on Drive so that they survive a disconnect
drive.mount("/content/drive")
os.environ["DRIVE_ROOT"] = DRIVE_DIR  # read by kaggle_ws.get_paths in validate.py

token = os.environ.get("GITHUB_TOKEN")
url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
if not os.path.exists(REPO_DIR):
    !git clone -q {url} {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH} && git log -1 --oneline

!pip install -q -U kaggle && pip install -q --no-deps -e .

sys.path.insert(0, f"{REPO_DIR}/src")
sys.modules.pop("kaggle_ws", None)
importlib.invalidate_caches()
from kaggle_ws.paths import drive_root

if not drive_root().is_dir():
    raise FileNotFoundError(f"No such Drive folder: {drive_root()}")
print("Drive folder:", drive_root() / COMPETITION)


In [ ]:
# @title Run
# stopping the cell by hand keeps the runtime, so you can rerun without setting up again
stopped = False
try:
    !python competitions/{COMPETITION}/validate.py --exps {EXPS} {ARGS}
    exit_code = _exit_code  # noqa: F821  (set by IPython after a ! command)
    stopped = exit_code in (-2, 130)  # SIGINT from the stop button
    if exit_code != 0:
        raise RuntimeError(f"validate.py failed (exit code {exit_code})")
except KeyboardInterrupt:
    stopped = True
    raise
finally:
    if DISCONNECT_WHEN_DONE and not stopped:
        from google.colab import runtime

        print("disconnecting the runtime; results are on Drive")
        runtime.unassign()
